In [ ]:
import numpy as np
import netket as nk
import jax
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import json
import flax.serialization as serialization
from scipy.ndimage import uniform_filter1d
import os
from tqdm import tqdm

In [ ]:
N = 12
N_ancilla = [4, 6, 8, 10]
hamiltonian_idx = 2


hamiltonian_file = f"../data/hamiltonians.json"
with open(hamiltonian_file, "r") as f:
    data_h = json.load(f)


J_ZZ = data_h['hamiltonian'][hamiltonian_idx]['J_ZZ']
J_XX = data_h['hamiltonian'][hamiltonian_idx]['J_XX']
h_x = data_h['hamiltonian'][hamiltonian_idx]['h_x']
h_z = data_h['hamiltonian'][hamiltonian_idx]['h_z']

print(f"J_ZZ = {J_ZZ}, J_XX = {J_XX}, h_x = {h_x}, h_z = {h_z}")

In [ ]:
from netket.operator.spin import sigmax, sigmaz

def hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended):
    H_system = 0
    H_extended = 0
    for i in range(N):
        H_system+= h_x * sigmax(hi_system, i)
        H_system+= h_z * sigmaz(hi_system, i)
        H_system += J_ZZ * sigmaz(hi_system, i) @ sigmaz(hi_system, (i + 1) % N)
        H_system += J_XX * sigmax(hi_system, i) @ sigmax(hi_system, (i + 1) % N)
        H_extended += h_x * sigmax(hi_extended, i)
        H_extended += h_z * sigmaz(hi_extended, i)
        H_extended += J_ZZ * sigmaz(hi_extended, i) @ sigmaz(hi_extended, (i + 1) % N)
        H_extended += J_XX * sigmax(hi_extended, i) @ sigmax(hi_extended, (i + 1) % N)
    return H_system, H_extended

hi_system = nk.hilbert.Spin(s=1/2, N=N)
hi_extended_full = nk.hilbert.Spin(s=1/2, N=2*N)
H_system_full, _ = hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended_full)
H_matrix = H_system_full.to_dense()
eigvals, eigvecs = np.linalg.eigh(H_matrix)

In [ ]:
save = False
fontsize = 14

if save:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "figure.titlesize": fontsize,
    })

    def save_fig(fig, name, subdir="plots/MRE"):
        base = os.path.join("..", subdir)
        os.makedirs(base, exist_ok=True)
        for ext in ("pgf", "pdf"):
            fig.savefig(os.path.join(base, f"{name}.{ext}"), bbox_inches="tight")
        print(f"Guardado: {base}/{name}.[pgf|pdf]")


fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].set_ylabel(r'$F$')
axes[0].set_title(r'(a) Free energy')
axes[0].set_xlabel(r'$T$')
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel(r'$\langle H\rangle$')
axes[1].set_title(r'(b) Energy')
axes[1].set_xlabel(r'$T$')
axes[1].grid(True, alpha=0.3)

axes[2].set_ylabel(r'$S$')
axes[2].set_title(r'(c) Entropy')
axes[2].set_xlabel(r'$T$')
axes[2].grid(True, alpha=0.3)

colormap = plt.cm.viridis_r
norm = mcolors.Normalize(vmin=min(N_ancilla), vmax=max(N_ancilla) + 1)

all_energy_results = []
all_entropy_results = []
all_free_energy_results = []
all_T_arrays = []

for N_A in N_ancilla:
    data_file = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/results_N{N}_vs_T.json"
    with open(data_file, "r") as f:
        data = json.load(f)
    T_array = np.array(data["T"])
    energy_vals = data["energy"]
    entropy_vals = data["entropy"]
    free_energy_vals = data["free_energy"]
    
    all_T_arrays.append(T_array)
    all_energy_results.append(energy_vals)
    all_entropy_results.append(entropy_vals)
    all_free_energy_results.append(free_energy_vals)

    hi_extended = nk.hilbert.Spin(s=1/2, N=N+N_A)
    H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended)

    color = colormap(norm(N_A))
    axes[0].plot(T_array, free_energy_vals, 'x', color=color, markersize=5, alpha=0.9,
                label=r'$N_a=%d$' % N_A)
    axes[1].plot(T_array, energy_vals, 'x', color=color, markersize=5, alpha=0.9,
                label=r'$N_a=%d$' % N_A)
    axes[2].plot(T_array, entropy_vals, 'x', color=color, markersize=5, alpha=0.9,
                label=r'$N_a=%d$' % N_A)

data_file = f"../data/{hamiltonian_idx}/N{N}/results_N{N}_vs_T.json"
with open(data_file, "r") as f:
    data = json.load(f)
T_array_mre = np.array(data["T"])
energy_mre = data["energy"]
entropy_mre = data["entropy"]
free_energy_mre = data["free_energy"]

hi_extended = nk.hilbert.Spin(s=1/2, N=2*N)
H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended)

color = colormap(norm(max(N_ancilla) + 1))
axes[0].plot(T_array_mre, free_energy_mre, 'x', color=color, markersize=5, alpha=0.9,
            label=r'$N_a=12$')
axes[1].plot(T_array_mre, energy_mre, 'x', color=color, markersize=5, alpha=0.9,
            label=r'$N_a=12$')
axes[2].plot(T_array_mre, entropy_mre, 'x', color=color, markersize=5, alpha=0.9,
            label=r'$N_a=12$')

axes[0].legend(fontsize=fontsize-1)
axes[1].legend(fontsize=fontsize-1)
axes[2].legend(fontsize=fontsize-1)
    
fig.tight_layout()
if save:
    save_fig(fig, f"F_E_S_N{N}_NA")
else: 
    plt.show()

In [ ]:
def check_temperatures(N, N_A):
    if N==N_A:
        data_file = f"../data/{hamiltonian_idx}/N{N}/results_N{N}_vs_T.json"
    else: 
        data_file = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/results_N{N}_vs_T.json"
    with open(data_file, "r") as f:
        data = json.load(f)
    T_list = data["T"]
    return np.array(T_list)

def load_params(N, N_A, T, vstate):
    if N==N_A:
        data_file = f"../data/{hamiltonian_idx}/N{N}/results_N{N}_vs_T.json"
    else:
        data_file = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/results_N{N}_vs_T.json"
    with open(data_file, "r") as f:
        data = json.load(f)
    T_list = data["T"]
    # buscar T más cercana
    idx = int(np.argmin(np.abs(np.array(T_list) - T)))
    file_idx = data["param_index"][idx]
    if N==N_A:
        filename = f"../data/{hamiltonian_idx}/N{N}/params/params_{file_idx:04d}.msgpack"
    else:
        filename = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/params/params_{file_idx:04d}.msgpack"
    with open(filename, "rb") as f:
        params = serialization.from_bytes(vstate.parameters, f.read())
    vstate.parameters = params

def canonical_expectation(O, T, eigvals, eigvecs):
    """
    Expectation value <O> en el ensemble canónico a temperatura T.

    Parámetros
    ----------
    O : NetKet operator
    T : float
    eigvals : ndarray
    eigvecs : ndarray

    Returns
    -------
    float
    """

    # convertir operador a matriz densa
    O_matrix = O.to_dense()

    # poblaciones de Boltzmann (estables numéricamente)
    log_pops = -eigvals / T
    log_pops -= np.max(log_pops)

    pops = np.exp(log_pops)
    pops /= pops.sum()

    # <n|O|n>
    O_diag = np.einsum(
        "ij,ji->i",
        np.conj(eigvecs.T),
        O_matrix @ eigvecs,
    )

    # promedio térmico
    return np.sum(pops * O_diag).real

def renyi_expectation(O, vstate):
    """Expectation value <O> en el ensemble de Rényi."""
    return float(vstate.expect(O).mean.real)

def renyi_exact_weights(T, eigvals, tol=1e-12):
    """
    Devuelve los pesos exactos w_k del MRE (alpha=2).
    """

    from scipy.optimize import brentq

    eigvals = np.array(eigvals, dtype=float)

    def mre_weights(E_perp):
        w = np.maximum(0.0, E_perp - eigvals)
        Z = w.sum()
        if Z < tol:
            return None
        return w / Z

    def f_root(E_perp):
        w = mre_weights(E_perp)
        if w is None:
            return E_perp
        E_bar = np.dot(w, eigvals)
        return E_perp - (2 * T + E_bar)

    E_min = eigvals.min()
    E_max = eigvals.max() + 4 * T

    try:
        E_perp = brentq(f_root, E_min, E_max, xtol=1e-12)
        w = mre_weights(E_perp)

    except ValueError:
        # ground state
        w = np.zeros(len(eigvals))
        w[0] = 1.0

    return w

def renyi_exact_expectation(O, T, eigvals, eigvecs):
    """
    Expectation value <O> en el MRE exacto (alpha=2).
    """

    w = renyi_exact_weights(T, eigvals)

    # operador en base de energía
    O_matrix = O.to_dense()

    O_diag = np.einsum(
        "ij,ji->i",
        np.conj(eigvecs.T),
        O_matrix @ eigvecs,
    ).real

    return float(np.dot(w, O_diag))

In [ ]:
save = True
fontsize = 14

N_ancilla = [4, 6, 8, 10, N]

temperatures = np.linspace(0.05, 4, 41)

if save:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "figure.titlesize": fontsize,
    })

    def save_fig(fig, name, subdir="plots/MRE"):
        import os
        base = os.path.join("..", subdir)
        os.makedirs(base, exist_ok=True)
        for ext in ("pgf", "pdf"):
            fig.savefig(os.path.join(base, f"{name}.{ext}"), bbox_inches="tight")
        print(f"Guardado: {base}/{name}.[pgf|pdf]")
else:
    import matplotlib.pyplot as plt

O_x = sum(sigmax(hi_system, i) for i in range(N)) / N
O_xx = sum(sigmax(hi_system, i) @ sigmax(hi_system, (i+1) % N) 
        for i in range(N)) / N
O_zz = sum(sigmaz(hi_system, i) @ sigmaz(hi_system, (i+1) % N) 
        for i in range(N)) / N

gibbs_sx, gibbs_sxx, gibbs_szz = [], [], []
exact_renyi_sx, exact_renyi_sxx, exact_renyi_szz = [], [], []

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].set_xlabel(r'$T$')
axes[0].set_ylabel(r'$\langle\sigma^x\rangle$')
axes[0].set_title(r'(a) $\langle\sigma^x\rangle$')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel(r'$T$')
axes[1].set_ylabel(r'$\Gamma^{x,x}$')
axes[1].set_title(r'(b) $\Gamma^{x,x}$')
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel(r'$T$')
axes[2].set_ylabel(r'$\Gamma^{z,z}$')
axes[2].set_title(r'(c) $\Gamma^{z,z}$')
axes[2].grid(True, alpha=0.3)

# Configurar colormap
colormap = plt.cm.viridis_r
norm = mcolors.Normalize(vmin=min(N_ancilla), vmax=max(N_ancilla) + 1)

for N_A in tqdm(N_ancilla, desc="Procesando vstates"):

    hi_extended = nk.hilbert.Spin(s=1/2, N=N+N_A)
    H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended)

    O_x_ext = sum(sigmax(hi_extended, i) for i in range(N)) / N
    O_xx_ext = sum(sigmax(hi_extended, i) @ sigmax(hi_extended, (i+1) % N)
                for i in range(N)) / N
    O_zz_ext = sum(sigmaz(hi_extended, i) @ sigmaz(hi_extended, (i+1) % N)
                for i in range(N)) / N

    sampler = nk.sampler.ARDirectSampler(hi_extended)
    model = nk.models.ARNNDense(hilbert=hi_extended, layers=1, features=16, activation=jax.nn.gelu)
    vstate = nk.vqs.MCState(sampler, model, n_samples=40960)

    T_array_compare = check_temperatures(N, N_A)

    renyi_sx, renyi_sxx, renyi_szz = [], [], []

    for T in T_array_compare:
        load_params(N, N_A, T, vstate)
        renyi_sx.append(renyi_expectation(O_x_ext, vstate))
        renyi_sxx.append(renyi_expectation(O_xx_ext, vstate))
        renyi_szz.append(renyi_expectation(O_zz_ext, vstate))

    # Obtener color basado en N_A
    color = colormap(norm(N_A))
    
    axes[0].scatter(T_array_compare, renyi_sx, s=10, zorder=3, 
                   color=color, label=r'$N_a=%d$' % N_A)
    axes[1].scatter(T_array_compare, renyi_sxx, s=10, zorder=3, 
                   color=color, label=r'$N_a=%d$' % N_A)
    axes[2].scatter(T_array_compare, renyi_szz, s=10, zorder=3, 
                   color=color, label=r'$N_a=%d$' % N_A)

# Opcional: Agregar línea para N_a = max(N_ancilla)+1 (como en tu segundo ejemplo)
# Descomenta si quieres incluir un caso adicional
"""
data_file = f"../data/{hamiltonian_idx}/N{N}/results_N{N}_vs_T.json"
with open(data_file, "r") as f:
    data = json.load(f)
T_array_mre = np.array(data["T"])
# ... cargar datos adicionales si los tienes

color = colormap(norm(max(N_ancilla) + 1))
axes[0].plot(T_array_mre, free_energy_mre, 'x', color=color, markersize=5, alpha=0.9,
            label=r'$N_a=%d$' % (max(N_ancilla) + 1))
# ... etc
"""

axes[0].legend(fontsize=fontsize-1)
axes[1].legend(fontsize=fontsize-1)
axes[2].legend(fontsize=fontsize-1)

plt.tight_layout()

if save:
    save_fig(fig, f"Observables_N{N}_NA")
else:
    plt.show()